# 03 — Selection reproducibility (dual view)

This notebook reproduces the **cross-platform Selection stage** from frozen Scopus/WoS unique CSVs.

Two views are kept **strictly separate**:

| View | Role |
|------|------|
| **HISTORICAL REPRODUCTION** | Reconstructs Paper-1 processing, including the WoS→Scopus reindex defect |
| **CORRECTED PIPELINE** | Intended schema alignment; reports how automatic candidates differ |

Manual screening and the recoverable final corpus come from preserved Excel evidence (`summary_reduced.xlsx`). They are **not** re-decided here.


## Source inputs

- `data/automatic/scopus/scopus_unique.csv` → 164
- `data/automatic/web_of_science/wos_unique.csv` → 62
- Concatenate → **226**


In [ ]:
from re_te_lowresources.selection import (
    ANCIENT_CHINESE_TITLE,
    EXPECTED_CORRECTED,
    EXPECTED_FINAL_CORPUS,
    EXPECTED_HISTORICAL,
    EXPECTED_MANUAL_DOUBT,
    EXPECTED_MANUAL_NO,
    EXPECTED_MANUAL_YES,
    PUBLISHED_FINAL_COUNT_WARNING,
    TERL_EIDS,
    WOS_TO_SCOPUS_RENAME,
    build_corrected_funnel,
    build_historical_funnel,
    format_funnel_arrow,
    load_platform_uniques,
    normalize_title,
    reproduce_selection,
)
from re_te_lowresources.scopus import default_repo_root

ROOT = default_repo_root()
scopus, wos = load_platform_uniques()
print("Repository root:", ROOT)
print("Scopus unique:", len(scopus))
print("WoS unique:", len(wos))


## Historical vs corrected schema alignment

**CORRECTED:** apply the intended WoS→Scopus rename map (including Researcher Ids / ORCIDs → `author(s) id`), then concatenate.

**HISTORICAL:** after lowercasing, reindex WoS onto Scopus column names **without** that rename — the documented defect that nulls WoS titles/language/author IDs.


In [ ]:
print("Intended rename keys (sample):")
for src, dst in list(WOS_TO_SCOPUS_RENAME.items())[:8]:
    print(f"  {src!r} → {dst!r}")
print(f"  … ({len(WOS_TO_SCOPUS_RENAME)} mappings total)")

historical = build_historical_funnel(scopus, wos)
corrected = build_corrected_funnel(scopus, wos)
print("\nHistorical WoS titled rows after merge:", historical.merged.loc[historical.merged["search_engine"]=="Web of Science", "title"].notna().sum())
print("Corrected WoS titled rows after merge:", corrected.merged.loc[corrected.merged["search_engine"]=="Web of Science", "title"].notna().sum())


## Side-by-side funnel


In [ ]:
import pandas as pd

rows = []
for step in ["merged", "unique", "english", "proceedings_filtered", "candidates"]:
    rows.append({
        "step": step,
        "HISTORICAL": historical.counts()[step],
        "CORRECTED": corrected.counts()[step],
        "delta": corrected.counts()[step] - historical.counts()[step],
        "hist_expected": EXPECTED_HISTORICAL[step],
        "corr_expected": EXPECTED_CORRECTED[step],
    })
funnel_table = pd.DataFrame(rows)
display(funnel_table)

print("Historical:", format_funnel_arrow(historical.counts()))
print("Corrected:", format_funnel_arrow(corrected.counts()))
assert historical.counts() == EXPECTED_HISTORICAL
assert corrected.counts() == EXPECTED_CORRECTED


## Record-level deltas (candidates)

Titles present in **CORRECTED** candidates but not in **HISTORICAL** candidates (normalized comparison).


In [ ]:
h_norm = set(historical.candidates["title"].map(normalize_title))
c_only = corrected.candidates[
    ~corrected.candidates["title"].map(normalize_title).isin(h_norm)
][["title", "doi", "search_engine", "document type"]]
display(c_only.reset_index(drop=True))
assert any("Ancient Chinese" in t for t in c_only["title"].astype(str))


## TERL normalization (135 → 134 historically)

Two Scopus bibliographic rows share one normalized title; candidate normalization keeps the first.

Methodology labels this −1 as “Free”; artifacts support TERL dual records, not free-access filtering.


In [ ]:
terl_proc = historical.proceedings_filtered[
    historical.proceedings_filtered["title"].astype(str).str.contains("TERL:", case=False, na=False)
][["title", "doi", "eid", "year"]]
display(terl_proc.reset_index(drop=True))
print("TERL EIDs expected:", sorted(TERL_EIDS))
print("TERL rows after candidate normalization:",
      historical.candidates["title"].astype(str).str.contains("TERL:", case=False, na=False).sum())


## Historical manual screening + final corpus

Exports are written by `reproduce_selection` (same shared implementation as the CLI).

> **WARNING:** publication reports **43**; preserved analytical artifacts contain **42**.


In [ ]:
result = reproduce_selection(ROOT, validate=True, write=True)

study = pd.read_csv(result.study_selection_path)
final_sel = pd.read_csv(result.final_selection_path)
corpus = pd.read_csv(result.final_corpus_path)

print("Manual choosen_state:", study["choosen_state"].value_counts().to_dict())
print(f"Yes {EXPECTED_MANUAL_YES} / No {EXPECTED_MANUAL_NO} / Doubt {EXPECTED_MANUAL_DOUBT}")
print("Final selection rows:", len(final_sel),
      "| in final corpus:", int(final_sel["in_final_corpus"].sum()))
print("Non-final Yes paper_ids:",
      sorted(final_sel.loc[~final_sel["in_final_corpus"], "paper_id"].astype(str)))
print("Final corpus:", len(corpus), "(expected", EXPECTED_FINAL_CORPUS, ")")
print()
print("WARNING:")
print(PUBLISHED_FINAL_COUNT_WARNING)
print("PASS")
